# Jupyter Notebook to clean data.xlsx

In [1]:
import pandas as pd
import numpy as np

## Loading the datasets

In [2]:
# replace with your Excel file path
file_path = "/Users/dazedinthecity/Documents/GitHub/CEU-DMDB-GroupH/data/private/data.xlsx"

# read first sheet into DataFrame
df = pd.read_excel(file_path, sheet_name=0)

# quick check
print(f"df shape: {df.shape}")
df.shape

FileNotFoundError: [Errno 2] No such file or directory: '/Users/dazedinthecity/Documents/GitHub/CEU-DMDB-GroupH/data/private/data.xlsx'

In [ ]:
# replace with your Excel file path
file_path_TE = "/Users/dazedinthecity/Desktop/Databases/TE_Course_Data_25-26.csv"

# read CSV into DataFrame
df_TE = pd.read_csv(file_path_TE)

# quick check
print(f"df_TE shape: {df_TE.shape}")
df_TE.shape

## Step 1: Combine df and df_TE

In [ ]:
# Show common and unique columns
common_cols = df.columns.intersection(df_TE.columns).tolist()
only_df = df.columns.difference(df_TE.columns).tolist()
only_df_TE = df_TE.columns.difference(df.columns).tolist()

print(f"Common columns ({len(common_cols)}): {common_cols}")
print(f"\nColumns only in df ({len(only_df)}): {only_df}")
print(f"\nColumns only in df_TE ({len(only_df_TE)}): {only_df_TE}")

# Concatenate (outer alignment of columns)
combined_df = pd.concat([df, df_TE], ignore_index=True, sort=False)
print(f"\nCombined dataframe shape: {combined_df.shape}")
print(f"Total rows: {len(df)} + {len(df_TE)} = {len(combined_df)}")
print(f"Total columns: {combined_df.shape[1]}")

# Quick peek
combined_df.head()

## Step 2: Remove rows with NAs >= 20%

In [ ]:
# Calculate the percentage of NAs for each row
na_threshold_row = 0.50  # 20%
total_cols = combined_df.shape[1]

# Count NAs per row
na_counts_per_row = combined_df.isna().sum(axis=1)
na_pct_per_row = na_counts_per_row / total_cols

# Identify rows to keep (those with < 20% NAs)
rows_to_keep = na_pct_per_row < na_threshold_row

print(f"Original number of rows: {len(combined_df)}")
print(f"Rows with >= 20% NAs: {(~rows_to_keep).sum()}")
print(f"Rows with < 20% NAs (to keep): {rows_to_keep.sum()}")

# Filter the dataframe
df_cleaned_rows = combined_df[rows_to_keep].copy()

print(f"\nDataframe shape after row cleaning: {df_cleaned_rows.shape}")
print(f"Rows removed: {len(combined_df) - len(df_cleaned_rows)}")

## Step 3: Remove columns with NAs >= 50%

In [ ]:
# Calculate the percentage of NAs for each column
na_threshold_col = 0.50  # 50%
total_rows = df_cleaned_rows.shape[0]

# Count NAs per column
na_counts_per_col = df_cleaned_rows.isna().sum()
na_pct_per_col = na_counts_per_col / total_rows

# Identify columns to keep (those with < 50% NAs)
cols_to_keep = na_pct_per_col < na_threshold_col
cols_to_drop = na_pct_per_col >= na_threshold_col

print(f"Original number of columns: {len(df_cleaned_rows.columns)}")
print(f"Columns with >= 50% NAs: {cols_to_drop.sum()}")
print(f"Columns with < 50% NAs (to keep): {cols_to_keep.sum()}")

# Show which columns will be dropped
if cols_to_drop.sum() > 0:
    print(f"\nColumns to be dropped ({cols_to_drop.sum()}):")
    for col in df_cleaned_rows.columns[cols_to_drop]:
        pct = na_pct_per_col[col] * 100
        print(f"  - {col}: {pct:.1f}% NAs")

# Filter the dataframe
df_final = df_cleaned_rows.loc[:, cols_to_keep].copy()

print(f"\nFinal dataframe shape: {df_final.shape}")
print(f"Columns removed: {len(df_cleaned_rows.columns) - len(df_final.columns)}")

## Summary of data cleaning

In [ ]:
print("="*60)
print("DATA CLEANING SUMMARY")
print("="*60)
print(f"\n1. Combined datasets:")
print(f"   - df: {df.shape}")
print(f"   - df_TE: {df_TE.shape}")
print(f"   - Combined: {combined_df.shape}")

print(f"\n2. Row cleaning (removed rows with >= 20% NAs):")
print(f"   - Before: {len(combined_df)} rows")
print(f"   - After: {len(df_cleaned_rows)} rows")
print(f"   - Removed: {len(combined_df) - len(df_cleaned_rows)} rows")

print(f"\n3. Column cleaning (removed columns with >= 50% NAs):")
print(f"   - Before: {len(df_cleaned_rows.columns)} columns")
print(f"   - After: {len(df_final.columns)} columns")
print(f"   - Removed: {len(df_cleaned_rows.columns) - len(df_final.columns)} columns")

print(f"\n4. Final dataset:")
print(f"   - Shape: {df_final.shape}")
print(f"   - Total cells: {df_final.shape[0] * df_final.shape[1]:,}")
print(f"   - Total NAs: {df_final.isna().sum().sum():,}")
print(f"   - Overall NA percentage: {(df_final.isna().sum().sum() / (df_final.shape[0] * df_final.shape[1]) * 100):.2f}%")
print("="*60)

## Preview of final cleaned data

In [ ]:
# Display first few rows
df_final.head()

In [ ]:
# Display info about the final dataframe
df_final.info()

In [ ]:
columns_to_drop = [
    'Taught',
    'Course occurence (code)',
    'UID',
    'External ID',
    'Terminated',
    'Ignore in TE Curriculum',
    'Import date','Course occurence',
    'Course occurence (diff)',
    'Temporary course code',
    'Publication date',
    'Assessment type',
    'Course offered for cross-listing',
    'Changes in Learning outcomes (EN)',
    'Changes in Learning activities and teaching methods (EN)',
    'Changes in Assessment (EN)',
    'Changes in Course contents (EN)',
    'Changes in Background and overall aim (EN)',
    'Changes in Contact details (EN)',
    'Changes in Course prerequisites (EN)',
    'Changes in Waiting list handling (EN)',
    'Load division (count)',
    'Course late changes (Winter term)',
    'Course detail late changes',
    'Course details',
    'Manage course basics',
    'Instructor (changes)'
    ]
cols_found = [col for col in columns_to_drop if col in df_final.columns]

if cols_found:
    df_final = df_final.drop(columns=cols_found)
    print(f"Dropped columns: {cols_found}")
    print(f"New shape: {df_final.shape}")
else:
    print(f"Columns not found in df_final")
    print(f"Available columns: {df_final.columns.tolist()}")

## Export cleaned data (optional)

In [ ]:
# Uncomment to save the cleaned data
output_path = "cleaned_data.xlsx"
df_final.to_excel(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")

